# An Agent With Its Own Wallet and a Budget

Pay for inference with a wallet signature instead of an API key, and cap what the agent can spend.

This notebook accompanies [Giving an Agent a Wallet and a Budget](https://docs.venice.ai/learn/wallet-budget-agent), which explains the reasoning behind each step. Run the cells in order.

**You do not need a funded wallet to run this.** Without one the notebook generates a disposable address, signs in with it, reads a zero balance, and stops at the payment wall. Every step except the payment itself is real.

## Setup

If you do want to spend, put a funded wallet's private key in the Colab sidebar under the key icon, as a secret named `WALLET_KEY`. Leave it unset to run unfunded.

The wallet needs at least five dollars of USDC on Base, which is the minimum top-up Venice will settle.

In [ ]:
%pip install -q "x402[evm]" eth-account requests

import os

try:
    from google.colab import userdata

    # Absent secret raises, which leaves the notebook on the disposable path.
    os.environ['WALLET_KEY'] = userdata.get('WALLET_KEY')
    print('Funded wallet key loaded.')
except Exception:
    print('No WALLET_KEY secret. Running with a disposable wallet.')

## Configuration

In [ ]:
import base64
import json
import os
import secrets
from datetime import datetime, timedelta, timezone

import requests
from eth_account import Account
from eth_account.messages import encode_defunct

BASE_URL = "https://api.venice.ai/api/v1"
DOMAIN = "api.venice.ai"
CHAIN_ID = 8453          # Base mainnet
MODEL = "qwen3-5-9b"
BUDGET_USD = 5.00

## The wallet

In production this is a wallet you funded deliberately, with the key in a secret manager. While building, a throwaway is safer, because a wallet with no money cannot do anything expensive by accident.

In [ ]:
key = os.environ.get("WALLET_KEY")
account = Account.from_key(key) if key else Account.create()

print(f"wallet {account.address}")
print("funded" if key else "disposable, cannot pay yet")

## Signing in

There is no key to send, so every request carries a signed [EIP-4361](https://eips.ethereum.org/EIPS/eip-4361) message proving the wallet owner made it. Venice rebuilds these exact bytes and verifies your signature against them, so the format is not negotiable.

Signatures last five minutes and each nonce is single use, so we sign a fresh header per request. Signing is local and costs nothing.

In [ ]:
def siwx_header():
    now = datetime.now(timezone.utc)
    stamp = lambda t: t.isoformat(timespec="milliseconds").replace("+00:00", "Z")
    issued_at, expires_at = stamp(now), stamp(now + timedelta(minutes=4))
    nonce = secrets.token_hex(8)

    message = (
        f"{DOMAIN} wants you to sign in with your Ethereum account:\n"
        f"{account.address}\n\nSign in to Venice AI\n\n"
        f"URI: https://{DOMAIN}\nVersion: 1\nChain ID: {CHAIN_ID}\n"
        f"Nonce: {nonce}\nIssued At: {issued_at}\nExpiration Time: {expires_at}"
    )
    signature = account.sign_message(encode_defunct(text=message)).signature.hex()

    payload = {
        "address": account.address,
        "message": message,
        "signature": signature if signature.startswith("0x") else "0x" + signature,
        "chainId": CHAIN_ID,
    }
    return base64.b64encode(json.dumps(payload).encode()).decode()

Read the balance back. `canConsume` already accounts for the ten cent floor.

In [ ]:
def wallet_get(path, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers={"SIGN-IN-WITH-X": siwx_header()},
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["data"]


def balance():
    return wallet_get(f"/x402/balance/{account.address}")


print(balance())

## Putting money in

Two requests: discover what Venice accepts, then settle a signed USDC transfer. The cell below only defines the function.

In [ ]:
from x402.client import SpendControls, x402ClientSync
from x402.http import PAYMENT_SIGNATURE_HEADER, encode_payment_signature_header
from x402.mechanisms.evm import EthAccountSigner
from x402.mechanisms.evm.exact.client import ExactEvmScheme
from x402.schemas.payments import PaymentRequired


def top_up():
    discovery = requests.post(f"{BASE_URL}/x402/top-up", timeout=30)
    required = PaymentRequired.model_validate(discovery.json())

    rail = next(a for a in required.accepts if a.network.startswith("eip155"))
    print(f"{rail.network}: {int(rail.amount) / 1e6:.2f} USDC to {rail.payTo}")

    client = x402ClientSync()
    client.register(rail.network, ExactEvmScheme(EthAccountSigner(account)))
    # The SDK caps one payment at $1 by default, which is below the $5 minimum
    # top-up, so every rail gets rejected until this is raised.
    client.set_spend_controls(SpendControls(max_amount_per_payment="$5", allowed_assets=True))

    payload = client.create_payment_payload(required)
    settlement = requests.post(
        f"{BASE_URL}/x402/top-up",
        headers={PAYMENT_SIGNATURE_HEADER: encode_payment_signature_header(payload)},
        timeout=90,
    )
    return settlement.json()

Running `top_up()` moves real money, so it is commented out. Uncomment it once `WALLET_KEY` points at a wallet holding at least five dollars of USDC on Base.

From an empty wallet it returns `400 PAYMENT_VERIFICATION_FAILED`, which means the signature was fine and the transfer was not.

In [ ]:
# print(top_up())

## Paying per call

Ordinary inference that happens to carry a signature. Turning off the Venice system prompt is worth about seventeen hundred input tokens per call, which dwarfs the question itself.

In [ ]:
def ask(question):
    response = requests.post(
        f"{BASE_URL}/chat/completions",
        headers={"SIGN-IN-WITH-X": siwx_header(), "Content-Type": "application/json"},
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": question}],
            "max_completion_tokens": 150,
            "venice_parameters": {
                "include_venice_system_prompt": False,
                "disable_thinking": True,
            },
        },
        timeout=90,
    )
    if response.status_code == 402:
        body = response.json()
        raise RuntimeError(
            f"balance ${body.get('currentBalanceUsd', 0)} is under the "
            f"${body.get('minimumBalanceUsd')} floor. Minimum top-up is "
            f"${body['topUpInstructions']['minimumAmountUsd']}."
        )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"].strip()

## Reading what it spent

The ledger is authoritative, so ask what was charged rather than estimating from token counts.

In [ ]:
def charges():
    """Every debit against this wallet, newest first."""
    ledger = wallet_get(f"/x402/transactions/{account.address}", limit=100)
    return [t for t in ledger["transactions"] if t["type"] == "CHARGE"]

## The budgeted run

Before each call the agent checks what it has spent and declines work it cannot pay for. Unfunded, this stops immediately and tells you the minimum top-up.

In [ ]:
TASKS = [
    "Name one concrete tradeoff of vector search versus keyword search. One sentence.",
    "In one sentence, when is a bloom filter the wrong choice?",
    "Give one reason CRDTs are hard to debug in production. One sentence.",
    "What is one failure mode of exponential backoff without jitter? One sentence.",
    "Name one thing consistent hashing does not solve. One sentence.",
    "Why is p99 latency more useful than the mean? One sentence.",
]


def run(budget=BUDGET_USD):
    opening = balance()
    print(f"balance ${opening['balanceUsd']:.4f}, budget ${budget:.4f}")

    if not opening["canConsume"]:
        print(f"cannot transact yet, minimum top-up is ${opening['minimumTopUpUsd']}")
        return

    baseline = sum(abs(c["amount"]) for c in charges())
    spent = 0.0

    for number, task in enumerate(TASKS, 1):
        if spent >= budget:
            print(f"\nstopped before task {number}: ${spent:.6f} of ${budget:.4f} spent")
            return

        answer = ask(task)
        spent = sum(abs(c["amount"]) for c in charges()) - baseline
        print(f"\n{number}. {task}")
        print(f"   {answer}")
        print(f"   ${spent:.6f} spent, ${budget - spent:.6f} left")

    print(f"\nfinished all {len(TASKS)} tasks for ${spent:.6f}")
    if spent:
        print(f"at this rate ${budget:.2f} covers about {int(budget / (spent / len(TASKS))):,} calls")

In [ ]:
run()              # $5.00, finishes every task
run(budget=1e-5)   # stops partway, having spent about $0.000007 per call

## Next steps

- [Authentication](https://docs.venice.ai/guides/getting-started/authentication), both auth modes side by side
- [x402 top-up](https://docs.venice.ai/api-reference/endpoint/x402/top-up), the endpoint reference
- [Building an Audio Research Notebook](https://docs.venice.ai/learn/audio-research-notebook), a longer project to point this agent's budget at
- `venice-x402-client` on npm, which wraps catch-402, top-up, and retry for TypeScript